In [1]:
# Phase 1 – Real Data Collection | Cell 1: Setup + Inventory

from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")

OUT_DIR  = PROJECT / "outputs" / "phase1"
OUT_DATA = OUT_DIR / "data"
OUT_TAB  = OUT_DIR / "tables"
OUT_FIG  = OUT_DIR / "figures"
OUT_LOG  = OUT_DIR / "logs"

for d in [OUT_DATA, OUT_TAB, OUT_FIG, OUT_LOG]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 1 – Real Data Collection | Cell 1")
print("=" * 60)
print(f"Timestamp    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Project Root : {PROJECT}")
print(f"Outputs      : {OUT_DIR}")

sources = {
    "JST_xlsx": PROJECT / "data" / "raw" / "Macrohistory" / "JSTdatasetR6.xlsx",
    "BACI_edges": PROJECT / "data" / "raw" / "baci" / "trade_edges_total_2000_2020.parquet",
    "BACI_nodes": PROJECT / "data" / "raw" / "baci" / "node_attributes_trade_2000_2020.parquet",
    "BIS_long": PROJECT / "data" / "raw" / "bis_credit" / "bis_credit_long.parquet",
    "ICIO_dir": PROJECT / "data" / "raw" / "oecd_icio",
    "IMF_dir": PROJECT / "data" / "raw" / "imf",
    "Crosswalk": PROJECT / "outputs" / "phase0_alignment" / "data" / "country_crosswalk_jst18.pkl",
}

print("\n--- Source inventory ---")
rows = []
for name, p in sources.items():
    exists = p.exists()
    if exists and p.is_file():
        size_mb = p.stat().st_size / 1e6
        n = 1
    elif exists and p.is_dir():
        files = list(p.glob("*.csv")) + list(p.glob("*.parquet")) + list(p.glob("*.xlsx"))
        size_mb = sum(f.stat().st_size for f in files) / 1e6
        n = len(files)
    else:
        size_mb, n = 0.0, 0
    status = "FOUND" if exists else "MISSING"
    print(f"  {name:12s} | {status:7s} | {size_mb:8.2f} MB | items={n}")
    rows.append({"source": name, "path": str(p), "exists": exists, "size_mb": round(size_mb, 2), "n_items": n})

pd.DataFrame(rows).to_csv(OUT_TAB / "phase1_source_inventory.csv", index=False)
print(f"\nSaved → {OUT_TAB / 'phase1_source_inventory.csv'}")
print("\nCell 1 completed successfully.")

Phase 1 – Real Data Collection | Cell 1
Timestamp    : 2026-09-25 20:18:58
Project Root : D:\univercity\omid_project\economics_project
Outputs      : D:\univercity\omid_project\economics_project\outputs\phase1

--- Source inventory ---
  JST_xlsx     | FOUND   |     1.41 MB | items=1
  BACI_edges   | FOUND   |     9.44 MB | items=1
  BACI_nodes   | FOUND   |     0.18 MB | items=1
  BIS_long     | FOUND   |     0.04 MB | items=1
  ICIO_dir     | FOUND   |  2356.61 MB | items=29
  IMF_dir      | FOUND   |     0.00 MB | items=0
  Crosswalk    | FOUND   |     0.00 MB | items=1

Saved → D:\univercity\omid_project\economics_project\outputs\phase1\tables\phase1_source_inventory.csv

Cell 1 completed successfully.


In [2]:
# Phase 1 – Real Data Collection | Cell 2: Load JST + Crosswalk

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase1" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase1" / "tables"

print("=" * 60)
print("Phase 1 – Real Data Collection | Cell 2: JST + Crosswalk")
print("=" * 60)

# Crosswalk
cw = pd.read_pickle(PROJECT / "outputs" / "phase0_alignment" / "data" / "country_crosswalk_jst18.pkl")
print("Crosswalk (18 countries):")
print(cw.to_string(index=False))

# JST full
jst = pd.read_excel(PROJECT / "data" / "raw" / "Macrohistory" / "JSTdatasetR6.xlsx")
print(f"\nJST raw: {jst.shape}")
print(f"Years: {jst['year'].min()} – {jst['year'].max()}")
print(f"Countries: {sorted(jst['country'].unique())}")

# Attach iso2/iso3
jst = jst.merge(cw[["country_jst", "iso2", "iso3"]],
                left_on="country", right_on="country_jst", how="left")

# Core window for multi-source analysis
YEAR_MIN, YEAR_MAX = 2000, 2020
jst_core = jst[(jst["year"] >= YEAR_MIN) & (jst["year"] <= YEAR_MAX)].copy()

print(f"\nJST core ({YEAR_MIN}-{YEAR_MAX}): {jst_core.shape}")
print(f"Countries: {jst_core['iso3'].nunique()}")

# Save
jst.to_pickle(OUT_DATA / "jst_full.pkl")
jst_core.to_pickle(OUT_DATA / "jst_core_2000_2020.pkl")
jst_core.to_csv(OUT_TAB / "jst_core_2000_2020.csv", index=False)
cw.to_pickle(OUT_DATA / "country_crosswalk.pkl")
cw.to_csv(OUT_TAB / "country_crosswalk.csv", index=False)

# Summary table
summary = pd.DataFrame({
    "metric": ["rows_full", "rows_core", "countries", "year_min_full", "year_max_full",
               "year_min_core", "year_max_core", "n_columns"],
    "value": [len(jst), len(jst_core), jst_core["iso3"].nunique(),
              int(jst["year"].min()), int(jst["year"].max()),
              int(jst_core["year"].min()), int(jst_core["year"].max()),
              jst.shape[1]]
})
summary.to_csv(OUT_TAB / "jst_load_summary.csv", index=False)
print(summary.to_string(index=False))

print(f"\nSaved → {OUT_DATA / 'jst_core_2000_2020.pkl'}")
print("\nCell 2 completed successfully.")

Phase 1 – Real Data Collection | Cell 2: JST + Crosswalk
Crosswalk (18 countries):
country_jst iso3 iso2  in_ICIO  in_BACI  in_BIS  in_all
  Australia  AUS   AU     True     True    True    True
    Belgium  BEL   BE     True     True    True    True
     Canada  CAN   CA     True     True    True    True
Switzerland  CHE   CH     True     True    True    True
    Germany  DEU   DE     True     True    True    True
    Denmark  DNK   DK     True     True    True    True
      Spain  ESP   ES     True     True    True    True
    Finland  FIN   FI     True     True    True    True
     France  FRA   FR     True     True    True    True
         UK  GBR   GB     True     True    True    True
    Ireland  IRL   IE     True     True    True    True
      Italy  ITA   IT     True     True    True    True
      Japan  JPN   JP     True     True    True    True
Netherlands  NLD   NL     True     True    True    True
     Norway  NOR   NO     True     True    True    True
   Portugal  PRT   PT

In [3]:
# Phase 1 – Real Data Collection | Cell 3: BACI Trade Network Data

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase1" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase1" / "tables"

print("=" * 60)
print("Phase 1 – Real Data Collection | Cell 3: BACI")
print("=" * 60)

cw = pd.read_pickle(OUT_DATA / "country_crosswalk.pkl")
iso2_set = set(cw["iso2"])

# Full edges (2000-2020 already)
edges = pd.read_parquet(PROJECT / "data" / "raw" / "baci" / "trade_edges_total_2000_2020.parquet")
nodes = pd.read_parquet(PROJECT / "data" / "raw" / "baci" / "node_attributes_trade_2000_2020.parquet")

print(f"BACI edges raw : {edges.shape}")
print(f"BACI nodes raw : {nodes.shape}")

# Filter to JST18 internal trade
edges_18 = edges[
    (edges["source_iso2"].isin(iso2_set)) &
    (edges["target_iso2"].isin(iso2_set))
].copy()

nodes_18 = nodes[nodes["country"].isin(iso2_set)].copy()

print(f"\nBACI edges JST18: {edges_18.shape}")
print(f"  years   : {edges_18['year'].min()} – {edges_18['year'].max()}")
print(f"  sources : {edges_18['source_iso2'].nunique()}")
print(f"  targets : {edges_18['target_iso2'].nunique()}")
print(f"  total value (bn USD): {edges_18['weight_value_kusd'].sum() / 1e6:.2f}")

print(f"\nBACI nodes JST18: {nodes_18.shape}")
print(nodes_18.head(3).to_string(index=False))

# Yearly density
yearly = edges_18.groupby("year").agg(
    n_edges=("weight_value_kusd", "count"),
    total_value_kusd=("weight_value_kusd", "sum"),
    mean_value_kusd=("weight_value_kusd", "mean")
).reset_index()
print("\n--- Yearly trade summary (sample) ---")
print(yearly.head(5).to_string(index=False))
print("...")
print(yearly.tail(3).to_string(index=False))

# Save
edges_18.to_pickle(OUT_DATA / "baci_edges_jst18.pkl")
edges_18.to_csv(OUT_TAB / "baci_edges_jst18.csv", index=False)
nodes_18.to_pickle(OUT_DATA / "baci_nodes_jst18.pkl")
nodes_18.to_csv(OUT_TAB / "baci_nodes_jst18.csv", index=False)
yearly.to_csv(OUT_TAB / "baci_yearly_summary.csv", index=False)

print(f"\nSaved → {OUT_DATA / 'baci_edges_jst18.pkl'}")
print("\nCell 3 completed successfully.")

Phase 1 – Real Data Collection | Cell 3: BACI
BACI edges raw : (621576, 14)
BACI nodes raw : (4668, 8)

BACI edges JST18: (6426, 14)
  years   : 2000 – 2020
  sources : 18
  targets : 18
  total value (bn USD): 81986.27

BACI nodes JST18: (378, 8)
 year country  export_kusd  import_kusd  net_trade_kusd  trade_openness_kusd  is_exporter  is_importer
 2000      AU 6.509249e+07 6.988506e+07   -4.792570e+06         1.349775e+08            1            1
 2000      BE 1.526895e+08 1.827496e+08   -3.006010e+07         3.354391e+08            1            1
 2000      CA 2.690087e+08 2.314422e+08    3.756641e+07         5.004509e+08            1            1

--- Yearly trade summary (sample) ---
 year  n_edges  total_value_kusd  mean_value_kusd
 2000      306      2.624544e+09     8.576940e+06
 2001      306      2.534824e+09     8.283739e+06
 2002      306      2.600256e+09     8.497568e+06
 2003      306      2.907138e+09     9.500452e+06
 2004      306      3.348376e+09     1.094241e+07
.

In [4]:
# Phase 1 – Real Data Collection | Cell 4: BIS + ICIO inventory

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase1" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase1" / "tables"

print("=" * 60)
print("Phase 1 – Real Data Collection | Cell 4: BIS + ICIO")
print("=" * 60)

cw = pd.read_pickle(OUT_DATA / "country_crosswalk.pkl")
iso2_set = set(cw["iso2"])
iso3_set = set(cw["iso3"])
YEAR_MIN, YEAR_MAX = 2000, 2020

# ---------- BIS ----------
bis = pd.read_parquet(PROJECT / "data" / "raw" / "bis_credit" / "bis_credit_long.parquet")
bis = bis[bis["country"].isin(iso2_set)].copy()
bis = bis[(bis["year"] >= YEAR_MIN) & (bis["year"] <= YEAR_MAX)]

print(f"BIS filtered: {bis.shape}")
print(f"Countries: {sorted(bis['country'].unique())}")
print(f"Borrowers: {bis['borrowers'].unique()}")
print(f"Lenders  : {bis['lenders'].unique()}")
print(f"Valuation: {bis['valuation'].unique()}")

# Build clean annual series
# Prefer: P (private), A (all lenders), M (market), unit 770
bis["lenders_clean"] = bis["lenders"].astype(str).str.replace('"', "").str.strip()
bis_p = bis[(bis["borrowers"] == "P") & (bis["valuation"] == "M")].copy()
# keep rows where lender starts with A (All sectors)
bis_p = bis_p[bis_p["lenders_clean"].str.startswith("A") | (bis_p["lenders_clean"] == "A")]

bis_annual = bis_p.groupby(["year", "country"], as_index=False)["value"].mean()
bis_annual = bis_annual.rename(columns={"country": "iso2", "value": "credit_private_pct_gdp"})

# Also government credit if available
bis_g = bis[(bis["borrowers"] == "G") & (bis["valuation"] == "M")].copy()
bis_g = bis_g[bis_g["lenders_clean"].str.startswith("A") | (bis_g["lenders_clean"] == "A")]
bis_g_annual = bis_g.groupby(["year", "country"], as_index=False)["value"].mean()
bis_g_annual = bis_g_annual.rename(columns={"country": "iso2", "value": "credit_gov_pct_gdp"})

bis_panel = bis_annual.merge(bis_g_annual, on=["year", "iso2"], how="outer")
print(f"\nBIS annual panel: {bis_panel.shape}")
print(f"Countries: {bis_panel['iso2'].nunique()}")
print(bis_panel.head(6).to_string(index=False))

bis_panel.to_pickle(OUT_DATA / "bis_credit_panel.pkl")
bis_panel.to_csv(OUT_TAB / "bis_credit_panel.csv", index=False)

# ---------- ICIO inventory ----------
icio_dir = PROJECT / "data" / "raw" / "oecd_icio"
icio_files = sorted(icio_dir.glob("*_SML.csv"))
icio_years = [int(f.stem.split("_")[0]) for f in icio_files]
print(f"\nICIO files: {len(icio_files)} | years {min(icio_years)}–{max(icio_years)}")

# Quick structure from one year
sample_year = 2015 if 2015 in icio_years else icio_years[len(icio_years)//2]
sample = pd.read_csv(icio_dir / f"{sample_year}_SML.csv", nrows=5)
print(f"Sample year {sample_year}: shape preview cols={sample.shape[1]}")

# Extract country list from row labels of one full file
lab = pd.read_csv(icio_dir / f"{sample_year}_SML.csv", usecols=[0])
labels = lab.iloc[:, 0].astype(str)
icio_all_cty = sorted({x.split("_")[0] for x in labels if "_" in x})
icio_jst = [c for c in icio_all_cty if c in iso3_set]
print(f"ICIO total regions: {len(icio_all_cty)}")
print(f"ICIO ∩ JST18: {len(icio_jst)} → {icio_jst}")

icio_info = pd.DataFrame({
    "year": icio_years,
    "file": [f.name for f in icio_files]
})
icio_info.to_csv(OUT_TAB / "icio_file_list.csv", index=False)
pd.DataFrame({"iso3": icio_jst}).to_csv(OUT_TAB / "icio_jst18_countries.csv", index=False)

print(f"\nSaved BIS panel + ICIO inventory")
print("\nCell 4 completed successfully.")

Phase 1 – Real Data Collection | Cell 4: BIS + ICIO
BIS filtered: (6048, 10)
Countries: ['AU', 'BE', 'CA', 'CH', 'DE', 'DK', 'ES', 'FI', 'FR', 'GB', 'IE', 'IT', 'JP', 'NL', 'NO', 'PT', 'SE', 'US']
Borrowers: <ArrowStringArray>
['G', 'P']
Length: 2, dtype: str
Lenders  : <ArrowStringArray>
['A', '"B', 'domestic"']
Length: 3, dtype: str
Valuation: <ArrowStringArray>
['M', 'N']
Length: 2, dtype: str

BIS annual panel: (378, 4)
Countries: 18
 year iso2  credit_private_pct_gdp  credit_gov_pct_gdp
 2000   AU                135.3750              19.700
 2000   BE                105.6875                 NaN
 2000   CA                101.9375                 NaN
 2000   CH                112.2500              44.500
 2000   DE                     NaN              61.175
 2000   DK                155.9500              58.075

ICIO files: 28 | years 1995–2022
Sample year 2015: shape preview cols=4538
ICIO total regions: 81
ICIO ∩ JST18: 18 → ['AUS', 'BEL', 'CAN', 'CHE', 'DEU', 'DNK', 'ESP', 'FIN'

In [5]:
# Phase 1 – Real Data Collection | Cell 5: Final Summary + Master Inventory

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase1" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase1" / "tables"

print("=" * 60)
print("Phase 1 – Real Data Collection | Cell 5: Final Summary")
print("=" * 60)

# Load all phase-1 artifacts
jst_core = pd.read_pickle(OUT_DATA / "jst_core_2000_2020.pkl")
baci_edges = pd.read_pickle(OUT_DATA / "baci_edges_jst18.pkl")
baci_nodes = pd.read_pickle(OUT_DATA / "baci_nodes_jst18.pkl")
bis_panel = pd.read_pickle(OUT_DATA / "bis_credit_panel.pkl")
cw = pd.read_pickle(OUT_DATA / "country_crosswalk.pkl")

# ICIO file count
icio_files = sorted((PROJECT / "data" / "raw" / "oecd_icio").glob("*_SML.csv"))
icio_years = [int(f.stem.split("_")[0]) for f in icio_files]

master = pd.DataFrame([
    {
        "source": "JST Macrohistory",
        "role": "Macro panel (historical + core)",
        "rows": len(jst_core),
        "countries": int(jst_core["iso3"].nunique()),
        "year_min": int(jst_core["year"].min()),
        "year_max": int(jst_core["year"].max()),
        "key_file": "jst_core_2000_2020.pkl",
        "status": "READY"
    },
    {
        "source": "BACI Total Trade",
        "role": "Bilateral trade network edges",
        "rows": len(baci_edges),
        "countries": int(baci_edges["source_iso2"].nunique()),
        "year_min": int(baci_edges["year"].min()),
        "year_max": int(baci_edges["year"].max()),
        "key_file": "baci_edges_jst18.pkl",
        "status": "READY"
    },
    {
        "source": "BACI Node Attributes",
        "role": "Country export/import totals",
        "rows": len(baci_nodes),
        "countries": int(baci_nodes["country"].nunique()),
        "year_min": int(baci_nodes["year"].min()),
        "year_max": int(baci_nodes["year"].max()),
        "key_file": "baci_nodes_jst18.pkl",
        "status": "READY"
    },
    {
        "source": "BIS Credit",
        "role": "Private & government credit %GDP",
        "rows": len(bis_panel),
        "countries": int(bis_panel["iso2"].nunique()),
        "year_min": int(bis_panel["year"].min()),
        "year_max": int(bis_panel["year"].max()),
        "key_file": "bis_credit_panel.pkl",
        "status": "READY"
    },
    {
        "source": "OECD ICIO",
        "role": "Inter-country input-output matrices",
        "rows": len(icio_files),
        "countries": 18,
        "year_min": min(icio_years),
        "year_max": max(icio_years),
        "key_file": "oecd_icio/*_SML.csv",
        "status": "READY (raw matrices)"
    },
])

print("\n=== MASTER DATA INVENTORY ===")
print(master.to_string(index=False))

master.to_csv(OUT_TAB / "phase1_master_inventory.csv", index=False)
master.to_pickle(OUT_DATA / "phase1_master_inventory.pkl")

# Core country list for all next phases
core_countries = cw[["country_jst", "iso2", "iso3"]].copy()
core_countries.to_csv(OUT_TAB / "core_countries_jst18.csv", index=False)
core_countries.to_pickle(OUT_DATA / "core_countries_jst18.pkl")

print(f"\nCore countries saved: {len(core_countries)}")
print(f"Common analysis window: 2000 – 2020")
print(f"\nAll Phase 1 artifacts in:\n  {OUT_DATA}\n  {OUT_TAB}")
print("\nCell 5 completed successfully.")
print("PHASE 1 COMPLETED.")

Phase 1 – Real Data Collection | Cell 5: Final Summary

=== MASTER DATA INVENTORY ===
              source                                role  rows  countries  year_min  year_max               key_file               status
    JST Macrohistory     Macro panel (historical + core)   378         18      2000      2020 jst_core_2000_2020.pkl                READY
    BACI Total Trade       Bilateral trade network edges  6426         18      2000      2020   baci_edges_jst18.pkl                READY
BACI Node Attributes        Country export/import totals   378         18      2000      2020   baci_nodes_jst18.pkl                READY
          BIS Credit    Private & government credit %GDP   378         18      2000      2020   bis_credit_panel.pkl                READY
           OECD ICIO Inter-country input-output matrices    28         18      1995      2022    oecd_icio/*_SML.csv READY (raw matrices)

Core countries saved: 18
Common analysis window: 2000 – 2020

All Phase 1 artifacts i